# Phase 5 — Publishing & Documentation
### Push Model to Hugging Face Hub + GitHub Setup

This notebook:
1. Pushes LoRA adapters to Hugging Face Hub
2. Creates a complete model card
3. Generates README for GitHub repo
4. Creates final project summary


In [ ]:
import os
import json
from huggingface_hub import HfApi, create_repo
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

print("All imports successful!")


## Step 1 — Configuration

In [ ]:
HF_USERNAME   = "faltooz123"   
REPO_NAME     = "qwen1.5-sql-qlora-spider"
# ────────────────────────────────────────────────────────────

CONFIG = {
    "base_model"    : "Qwen/Qwen1.5-1.8B-Chat",
    "adapter_path"  : "./outputs/qwen-sql-qlora/final_adapter",
    "repo_id"       : f"{HF_USERNAME}/{REPO_NAME}",
    "comparison"    : "./data/comparison_results.json",
}

print(f"Hugging Face repo : {CONFIG['repo_id']}")
print(f"Adapter path      : {CONFIG['adapter_path']}")


## Step 2 — Load Evaluation Results

In [ ]:
# Load comparison results for model card
with open(CONFIG["comparison"], "r") as f:
    comparison = json.load(f)

baseline_em  = comparison["baseline"]["exact_match_accuracy"]
baseline_tm  = comparison["baseline"]["avg_token_match"]
finetuned_em = comparison["finetuned"]["exact_match_accuracy"]
finetuned_tm = comparison["finetuned"]["avg_token_match"]
mmlu_acc     = comparison.get("mmlu_accuracy", "not run")

print("Evaluation results loaded!")
print(f"  Baseline  Exact Match : {baseline_em}%")
print(f"  Finetuned Exact Match : {finetuned_em}%")
print(f"  Baseline  Token Match : {baseline_tm}%")
print(f"  Finetuned Token Match : {finetuned_tm}%")
print(f"  MMLU Accuracy         : {mmlu_acc}")


## Step 3 — Create Hugging Face Repository

In [ ]:
api = HfApi()

# Create repo on Hugging Face Hub
try:
    create_repo(
        repo_id  = CONFIG["repo_id"],
        repo_type= "model",
        exist_ok = True,
        private  = False,
    )
    print(f"Repository created: https://huggingface.co/{CONFIG['repo_id']}")
except Exception as e:
    print(f"Repo already exists or error: {e}")


## Step 4 — Create Model Card

In [ ]:
# Build model card in parts to avoid nested f-string issues

header = """---
language:
- en
license: apache-2.0
base_model: Qwen/Qwen1.5-1.8B-Chat
tags:
- text-to-sql
- sql
- qlora
- lora
- fine-tuned
- spider
datasets:
- xlangai/spider
pipeline_tag: text-generation
---"""

description = """
# Qwen1.5-1.8B SQL Fine-Tuned (QLoRA) on Spider Dataset

## Model Description

This model is a fine-tuned version of [Qwen/Qwen1.5-1.8B-Chat](https://huggingface.co/Qwen/Qwen1.5-1.8B-Chat)
adapted for **Text-to-SQL generation** using the Spider dataset.

Fine-tuning was done using **QLoRA** (Quantized Low-Rank Adaptation) — a
parameter-efficient method that trains only a small set of adapter weights
instead of the full model.

## Intended Use

Convert natural language questions into SQL queries.

**Example:**
- Input: "How many singers do we have?"
- Output: `SELECT count(*) FROM singer`

## Training Data

- **Dataset**: [Spider](https://huggingface.co/datasets/xlangai/spider)
- **Samples used**: 500 training samples (subset)
- **Format**: Qwen chat instruction format

## Training Procedure

### Hardware
- GPU: NVIDIA GeForce RTX 5060 Laptop GPU (8GB VRAM)
- Training time: ~15 minutes

### Hyperparameters
| Parameter | Value |
|---|---|
| Base model | Qwen/Qwen1.5-1.8B-Chat |
| LoRA rank (r) | 16 |
| LoRA alpha | 32 |
| LoRA dropout | 0.05 |
| Target modules | q_proj, v_proj, k_proj, o_proj |
| Learning rate | 2e-4 |
| Batch size | 8 |
| Gradient accumulation | 2 (effective batch: 16) |
| Epochs | 2 |
| LR scheduler | cosine |
| Quantization | 4-bit NF4 (QLoRA) |
| Max sequence length | 512 |
"""

eval_section = f"""
## Evaluation Results

### SQL Generation (Spider validation set, 200 samples)

| Metric | Baseline | Fine-Tuned | Improvement |
|---|---|---|---|
| Exact Match Accuracy | {baseline_em}% | {finetuned_em}% | +{round(finetuned_em - baseline_em, 2)}% |
| Avg Token Match | {baseline_tm}% | {finetuned_tm}% | +{round(finetuned_tm - baseline_tm, 2)}% |

### Catastrophic Forgetting Check (MMLU)
| Metric | Score |
|---|---|
| MMLU Accuracy (50 samples) | {mmlu_acc}% |
| Random baseline | 25.0% |

The model retains general knowledge after SQL fine-tuning.

## Limitations

- Trained on a subset (500 samples) of Spider — full dataset has 7,000+ samples
- May struggle with complex multi-table JOIN queries
- Best performance on simple SELECT, COUNT, GROUP BY queries
- Fine-tuned for ~15 minutes — longer training would improve results
"""

repo_id = CONFIG["repo_id"]

usage_section = f"""
## How to Load and Use

```python
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

base_model_name = "Qwen/Qwen1.5-1.8B-Chat"
adapter_name    = "{repo_id}"

tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)

bnb_config = BitsAndBytesConfig(
    load_in_4bit           = True,
    bnb_4bit_quant_type    = "nf4",
    bnb_4bit_compute_dtype = torch.float16,
)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config = bnb_config,
    device_map          = {{"": 0}},
    trust_remote_code   = True,
)

model = PeftModel.from_pretrained(base_model, adapter_name)
model.eval()

def generate_sql(question, db_id):
    prompt = (
        "<|im_start|>system\\n"
        "You are an expert SQL assistant.<|im_end|>\\n"
        "<|im_start|>user\\n"
        f"Database: {{db_id}}\\n"
        f"Question: {{question}}\\n"
        "Write only the SQL query.<|im_end|>\\n"
        "<|im_start|>assistant\\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

sql = generate_sql("How many singers do we have?", "concert_singer")
print(sql)
```

## Citation

```bibtex
@misc{{qwen1.5-sql-qlora,
  title = {{Qwen1.5-1.8B SQL Fine-Tuned with QLoRA on Spider}},
  year  = {{2025}},
}}
```
"""

# Combine all parts
model_card = header + description + eval_section + usage_section

# Save model card locally
with open("./outputs/qwen-sql-qlora/final_adapter/README.md", "w", encoding="utf-8") as f:
    f.write(model_card)

print("Model card created!")
print(f"Length: {len(model_card)} characters")

## Step 5 — Push Adapters to Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

print(f"Pushing to: https://huggingface.co/{CONFIG['repo_id']}")
print("This may take a few minutes...\n")

# ── Load tokenizer ────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["base_model"],
    trust_remote_code=True
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Tokenizer loaded!")

# ── Load base model (NO quantization — for pushing to hub) ────
print("Loading base model for pushing (no quantization)...")
base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"],
    torch_dtype       = torch.float16,
    device_map        = "cpu",          # load on CPU for pushing
    trust_remote_code = True,
)
print("Base model loaded on CPU!")

# ── Load LoRA adapter on top ──────────────────────────────────
print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(
    base_model,
    CONFIG["adapter_path"],
    device_map = "cpu",
)
print("Adapter loaded!")

# ── Push to Hugging Face Hub ──────────────────────────────────
print("\nPushing adapter to Hub...")
model.push_to_hub(CONFIG["repo_id"])

print("Pushing tokenizer to Hub...")
tokenizer.push_to_hub(CONFIG["repo_id"])

print(f"\n✅ Model pushed successfully!")
print(f"View at: https://huggingface.co/{CONFIG['repo_id']}")

In [ ]:
# Push model card (README.md) to hub
api.upload_file(
    path_or_fileobj = "./outputs/qwen-sql-qlora/final_adapter/README.md",
    path_in_repo    = "README.md",
    repo_id         = CONFIG["repo_id"],
    repo_type       = "model",
)

print("Model card pushed!")
print(f"\nYour model is now live at:")
print(f"https://huggingface.co/{CONFIG['repo_id']}")
